In [2]:
import pandas as pd
import numpy as np

In [3]:
data_path = "C:\\Users\\miots\\m-thesis\\m-thesis\\data\\output\\sige\\sige_ElectricalConductivity_curves.csv"

df = pd.read_csv(data_path)

In [4]:
print(df.head())

   Index    SID                         DOI composition  sample_id  figure_id  \
0  33237  18378  10.1109/green.2012.6200928  Si0.8Ge0.2      10279       8381   
1  35271   1205   10.1007/s10789-005-0256-0  Si0.4Ge0.6      10951       9039   
2  40632   8779    10.1109/ict.2002.1190281  Si0.5Ge0.5      12652      10466   
3  40633   8779    10.1109/ict.2002.1190281  Si0.5Ge0.5      12652      10467   
4  40634   8779    10.1109/ict.2002.1190281  Si0.5Ge0.5      12653      10468   

        prop_x                   prop_y unit_x           unit_y  ...  \
0  Temperature  Electrical conductivity      K  ohm^(-1)*m^(-1)  ...   
1  Temperature  Electrical conductivity      K  ohm^(-1)*m^(-1)  ...   
2  Temperature   Electrical resistivity      K            ohm*m  ...   
3  Temperature   Electrical resistivity      K            ohm*m  ...   
4  Temperature   Electrical resistivity      K            ohm*m  ...   

                                updated_at                project_names  \
0  Th

In [13]:
# composition=Si0.5Ge0.5, prop_y=electrical_resistivity, and y_list contains any negative value
import ast
import json

def parse_y_list(y_raw):
    if pd.isna(y_raw):
        return []

    text = str(y_raw).strip()
    if not text:
        return []

    parsed = None
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            break
        except (TypeError, ValueError, SyntaxError, json.JSONDecodeError):
            continue

    if not isinstance(parsed, (list, tuple)):
        return []

    values = []
    for v in parsed:
        if v is None:
            continue
        try:
            values.append(float(v))
        except (TypeError, ValueError):
            continue
    return values

prop_y_norm = (
    df['prop_y']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace('_', ' ', regex=False)
)

filtered_df = df[
    (df['composition'].astype(str).str.strip() == 'Si0.5Ge0.5')
    & (prop_y_norm == 'electrical resistivity')
    & (df['y_list'].apply(lambda x: any(v < 0 for v in parse_y_list(x))))
]

print(filtered_df)
print(f'rows: {len(filtered_df)}')


   Index   SID                       DOI composition  sample_id  figure_id  \
2  40632  8779  10.1109/ict.2002.1190281  Si0.5Ge0.5      12652      10466   
4  40634  8779  10.1109/ict.2002.1190281  Si0.5Ge0.5      12653      10468   
5  40635  8779  10.1109/ict.2002.1190281  Si0.5Ge0.5      12654      10469   
7  40637  8779  10.1109/ict.2002.1190281  Si0.5Ge0.5      12655      10471   

        prop_x                  prop_y unit_x unit_y  ...  \
2  Temperature  Electrical resistivity      K  ohm*m  ...   
4  Temperature  Electrical resistivity      K  ohm*m  ...   
5  Temperature  Electrical resistivity      K  ohm*m  ...   
7  Temperature  Electrical resistivity      K  ohm*m  ...   

                                updated_at                project_names  \
2  Sun Feb 10 2019 11:30:44 GMT+0900 (JST)  ["ThermoelectricMaterials"]   
4  Sun Feb 10 2019 11:34:28 GMT+0900 (JST)  ["ThermoelectricMaterials"]   
5  Sun Feb 10 2019 11:35:27 GMT+0900 (JST)  ["ThermoelectricMaterials"]   
7  

In [14]:
# Optional: inspect compact columns for extracted rows
print(filtered_df[['Index', 'SID', 'composition', 'prop_y', 'y_list']])

negative_counts = filtered_df['y_list'].apply(
    lambda x: sum(v < 0 for v in parse_y_list(x))
)
print('negative value count per row:')
print(negative_counts.to_string(index=False))


   Index   SID composition                  prop_y  \
2  40632  8779  Si0.5Ge0.5  Electrical resistivity   
4  40634  8779  Si0.5Ge0.5  Electrical resistivity   
5  40635  8779  Si0.5Ge0.5  Electrical resistivity   
7  40637  8779  Si0.5Ge0.5  Electrical resistivity   

                                              y_list  
2  [507.7106008958046, 477.14613172859544, 485.26...  
4  [167.36037542279414, 194.85030138470367, 185.7...  
5  [17774.557817881527, 10822.174679206393, 7777....  
7  [2047.0322024776867, 2045.5268827233983, 2194....  
negative value count per row:
 3
 8
14
 3
